<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractal002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fftn, fftshift

def initialize_field(N, seed):
    np.random.seed(seed)
    field = np.random.randn(N, N, N)
    damping = np.linspace(0, 1, N)
    damping_3d = np.outer(np.outer(damping, damping), damping).reshape(N, N, N)
    return field * damping_3d

def evolve_HES(field, dt=0.01, steps=100):
    for _ in range(steps):
        laplacian = np.roll(field, 1, axis=0) + np.roll(field, -1, axis=0) \
                  + np.roll(field, 1, axis=1) + np.roll(field, -1, axis=1) \
                  + np.roll(field, 1, axis=2) + np.roll(field, -1, axis=2) - 6 * field
        field += dt * laplacian
    return field

def compute_power_spectrum(field):
    spectrum = np.abs(fftshift(fftn(field)))**2
    return spectrum

def extract_metrics(spectrum):
    N = spectrum.shape[0]
    center = N // 2
    r = np.sqrt((np.indices(spectrum.shape) - center)**2).sum(axis=0)
    r = r.astype(int)
    radial_profile = np.bincount(r.ravel(), spectrum.ravel()) / np.bincount(r.ravel())
    peak_index = np.argmax(radial_profile)
    sharpness = radial_profile[peak_index] / np.sum(radial_profile)
    entropy = -np.sum(radial_profile * np.log(radial_profile + 1e-12)) / np.log(len(radial_profile))
    return peak_index, sharpness, entropy

def run_fractal002():
    scales = [90, 95, 96, 97, 98, 99, 100, 101, 102, 105, 110]
    seeds = range(10)
    results = {}

    for N in scales:
        λ_list, sharp_list, entropy_list = [], [], []
        for seed in seeds:
            field = initialize_field(N, seed)
            field = evolve_HES(field)
            spectrum = compute_power_spectrum(field)
            λ, sharp, entropy = extract_metrics(spectrum)
            λ_list.append(λ)
            sharp_list.append(sharp)
            entropy_list.append(entropy)

        λ_mean, λ_std = np.mean(λ_list), np.std(λ_list)
        sharp_mean = np.mean(sharp_list)
        entropy_mean = np.mean(entropy_list)

        results[N] = (λ_mean, λ_std, sharp_mean, entropy_mean)
        print(f"\nScale s = {N}")
        print(f"  λ_dom (mean ± std): {λ_mean:.2f} ± {λ_std:.2f}")
        print(f"  Spectral sharpness: {sharp_mean:.2f}")
        print(f"  Spectral entropy: {entropy_mean:.2f}")

    # Plotting
    scales_sorted = sorted(results.keys())
    λ_means = [results[s][0] for s in scales_sorted]
    sharpness = [results[s][2] for s in scales_sorted]
    entropy = [results[s][3] for s in scales_sorted]

    plt.figure(figsize=(6, 4))
    plt.plot(scales_sorted, λ_means, marker='o')
    plt.title("λ_dom vs. Scale")
    plt.xlabel("Scale (N)")
    plt.ylabel("λ_dom")
    plt.grid(True)
    plt.savefig("lambda_dom_vs_scale.png")
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.plot(scales_sorted, sharpness, marker='o', color='green')
    plt.title("Spectral Sharpness vs. Scale")
    plt.xlabel("Scale (N)")
    plt.ylabel("Sharpness")
    plt.grid(True)
    plt.savefig("sharpness_vs_scale.png")
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.plot(scales_sorted, entropy, marker='o', color='red')
    plt.title("Spectral Entropy vs. Scale")
    plt.xlabel("Scale (N)")
    plt.ylabel("Entropy")
    plt.grid(True)
    plt.savefig("entropy_vs_scale.png")
    plt.close()

run_fractal002()



Scale s = 90
  λ_dom (mean ± std): 1.00 ± 1.61
  Spectral sharpness: 0.17
  Spectral entropy: -876481.74

Scale s = 95
  λ_dom (mean ± std): 1.30 ± 1.73
  Spectral sharpness: 0.12
  Spectral entropy: -974307.33

Scale s = 96
  λ_dom (mean ± std): 2.20 ± 2.32
  Spectral sharpness: 0.12
  Spectral entropy: -1030944.75

Scale s = 97
  λ_dom (mean ± std): 1.80 ± 1.60
  Spectral sharpness: 0.09
  Spectral entropy: -994826.75

Scale s = 98
  λ_dom (mean ± std): 1.60 ± 1.43
  Spectral sharpness: 0.13
  Spectral entropy: -1094840.36

Scale s = 99
  λ_dom (mean ± std): 2.20 ± 1.66
  Spectral sharpness: 0.13
  Spectral entropy: -1199204.62

Scale s = 100
  λ_dom (mean ± std): 1.30 ± 1.19
  Spectral sharpness: 0.12
  Spectral entropy: -1232515.52

Scale s = 101
  λ_dom (mean ± std): 1.90 ± 1.81
  Spectral sharpness: 0.14
  Spectral entropy: -1378408.36

Scale s = 102
  λ_dom (mean ± std): 1.90 ± 1.70
  Spectral sharpness: 0.12
  Spectral entropy: -1268088.51

Scale s = 105
  λ_dom (mean ± std): 